In [ ]:
import gc
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import ElasticNet
from tqdm import tqdm
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import pickle

In [ ]:
with open("data/characteristics_with_ind_interaction_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
import os
memory_info = os.popen('free -h').read()
print(memory_info)

# Enet

In [ ]:
import gc
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import ElasticNet
from tqdm import tqdm
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import os

# 加载数据
with open("data/characteristics_with_ind_interaction_return.pkl", "rb") as f:
    data = pickle.load(f)

import os
memory_info = os.popen('free -h').read()
print(memory_info)

def train_enet(data):
    # 60年数据
    start_year_train = 1957
    start_year_test = 2006
    end_year_test = 2016
    
    # 将 'DATE' 列转换为日期格式（如果尚未转换）
    data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
    
    # 用于累积所有年份的预测和真实值
    overall_pred_y_list = []
    overall_true_y_list = []
    
    # 循环处理每一年
    for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='year'):
        # 定义训练和测试的起止日期
        start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
        end_date_train = pd.Timestamp(year=year-13, month=12, day=31)
    
        start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
        end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
        
        start_date_test = pd.Timestamp(year=year, month=1, day=1)
        end_date_test = pd.Timestamp(year=year, month=12, day=31)
        
        print(f'训练开始日期: {start_date_train.strftime("%Y-%m")}')
        print(f'训练结束日期: {end_date_train.strftime("%Y-%m")}')
        print(f'验证开始日期: {start_date_val.strftime("%Y-%m")}')
        print(f'验证结束日期: {end_date_val.strftime("%Y-%m")}')
        print(f'测试开始日期: {start_date_test.strftime("%Y-%m")}')
        print(f'测试结束日期: {end_date_test.strftime("%Y-%m")}')
    
        # 选择训练和测试数据
        train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
        test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
        
        # 提取特征和目标变量
        X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
        y_train = train_data['RET'].values  
        X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
        y_test = test_data['RET'].values
        
        # 标准化特征
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        # 训练弹性网络模型
        enet_model = ElasticNet(alpha=0.5, l1_ratio=0.5)
        enet_model.fit(X_train, y_train)
       
        # 使用弹性网络进行预测
        y_test_pred_enet = enet_model.predict(X_test)
        
        pred_y_list.extend(y_test_pred_enet)
        true_y_list.extend(y_test)
        tmp=pd.DataFrame({'pred':[i[0] for i in np.array(y_test_pred)],'true':[i[0] for i in np.array(y_test)]})
        tmp.to_csv(f'output/{year}.csv')
        gc.collect()

        # 累积所有年份的预测和真实值
        overall_pred_y_list.extend(y_test_pred_enet)
        overall_true_y_list.extend(y_test)

train_enet(data)

# GLM

In [ ]:
import gc
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import statsmodels.api as sm
import os

# 加载数据
with open("data/characteristics_with_ind_interaction_return.pkl", "rb") as f:
    data = pickle.load(f)

# 显示内存信息
memory_info = os.popen('free -h').read()
print(memory_info)

def train_glm(data):
    # 60年数据
    start_year_train = 1957
    start_year_test = 1987
    end_year_test = 2016
    
    # 将 'DATE' 列转换为日期格式（如果尚未转换）
    data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
    
    # 用于累积所有年份的预测和真实值
    overall_pred_y_list = []
    overall_true_y_list = []
    
    # 循环处理每一年
    for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='year'):
        # 定义训练和测试的起止日期
        start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
        end_date_train = pd.Timestamp(year=year-13, month=12, day=31)
    
        start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
        end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
        
        start_date_test = pd.Timestamp(year=year, month=1, day=1)
        end_date_test = pd.Timestamp(year=year, month=12, day=31)
        
        print(f'训练开始日期: {start_date_train.strftime("%Y-%m")}')
        print(f'训练结束日期: {end_date_train.strftime("%Y-%m")}')
        print(f'验证开始日期: {start_date_val.strftime("%Y-%m")}')
        print(f'验证结束日期: {end_date_val.strftime("%Y-%m")}')
        print(f'测试开始日期: {start_date_test.strftime("%Y-%m")}')
        print(f'测试结束日期: {end_date_test.strftime("%Y-%m")}')
    
        # 选择训练和测试数据
        train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
        test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
        
        # 提取特征和目标变量
        X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
        y_train = train_data['RET'].values  
        X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
        y_test = test_data['RET'].values
        
        # 标准化特征
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        # 添加截距项
        X_train = sm.add_constant(X_train)
        X_test = sm.add_constant(X_test)
        # 训练GLM模型
        glm_model = sm.GLM(y_train, X_train, family=sm.families.Gaussian())
        glm_results = glm_model.fit()
       
        # 使用GLM进行预测
        y_test_pred_glm = glm_results.predict(X_test)
        
        # 保存预测结果到CSV文件
        tmp = pd.DataFrame({'pred': y_test_pred_glm, 'true': y_test})
        tmp.to_csv(f'GLM/{year}.csv', index=False)
        
        # 累积所有年份的预测和真实值
        overall_pred_y_list.extend(y_test_pred_glm)
        overall_true_y_list.extend(y_test)

train_glm(data)

In [ ]:
# def train_enet(data):
#     # 60年数据
#     start_year_train = 1957
#     start_year_test = 1987
#     end_year_test = 2016
    
#     # 将 'DATE' 列转换为日期格式（如果尚未转换）
#     data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
    
#     pred_y_list = []
#     true_y_list = []
    
#     # 循环处理每一年
#     for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
#         # 定义训练和测试的起止日期
#         start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
#         end_date_train = pd.Timestamp(year=year-13, month=12, day=31)
    
#         start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
#         end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
        
#         start_date_test = pd.Timestamp(year=year, month=1, day=1)
#         end_date_test = pd.Timestamp(year=year, month=12, day=31)
        
#         print(f'训练开始日期: {start_date_train.strftime("%Y-%m")}')
#         print(f'训练结束日期: {end_date_train.strftime("%Y-%m")}')
#         print(f'验证开始日期: {start_date_val.strftime("%Y-%m")}')
#         print(f'验证结束日期: {end_date_val.strftime("%Y-%m")}')
#         print(f'测试开始日期: {start_date_test.strftime("%Y-%m")}')
#         print(f'测试结束日期: {end_date_test.strftime("%Y-%m")}')
    
#         # 选择训练和测试数据
#         train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
#         test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
        
#         # 提取特征和目标变量
#         X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
#         y_train = train_data['RET'].values  
#         X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
#         y_test = test_data['RET'].values
        
#         # 标准化特征
#         scaler = StandardScaler()
#         X_train = scaler.fit_transform(X_train)
#         X_test = scaler.transform(X_test)
        
#         # 训练弹性网络模型
#         enet_model = ElasticNet(alpha=1.0, l1_ratio=0.5)
#         enet_model.fit(X_train, y_train)
       
#         # 使用弹性网络进行预测
#         y_test_pred_enet = enet_model.predict(X_test)
    
#         pred_y_list.extend(y_test_pred_enet)
#         true_y_list.extend(y_test)
        
#         # 输出每年的 R² 分数
#         print('Elastic Net R^2:{}'.format(r2_score(y_test, y_test_pred_enet) * 100))
        
#     # 计算总体 R² 分数
#     print('Overall R^2:{}'.format(r2_score(np.array(true_y_list), np.array(pred_y_list)) * 100))

In [ ]:
# import pandas as pd
# import numpy as np
# import pickle
# from sklearn.linear_model import ElasticNet
# from tqdm import tqdm
# from sklearn.metrics import r2_score
# import os

# def train_ols(data):
#     start_year_train = 1957
#     start_year_test = 1987
#     end_year_test = 2016
    
#     # Convert the 'Date' column to datetime if it's not already
#     data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
    
#     # Loop over the years
#     for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
#         # Define the start and end dates for training and test
#         start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
#         end_date_train = pd.Timestamp(year=year-13, month=12, day=31)
    
#         start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
#         end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
        
#         start_date_test = pd.Timestamp(year=year, month=1, day=1)
#         end_date_test = pd.Timestamp(year=year, month=12, day=31)
        
#         print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
#         print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    
#         print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
#         print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
        
#         print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
#         print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
#         # Select the data for training and test
#         train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
#         test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
        
#         # Extract features and target variables
#         X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
#         y_train = train_data['RET'].values  
#         X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
#         y_test = test_data['RET'].values
#         print(X_test)
#         # 使用Elastic Net模型
#         model = ElasticNet(alpha=1.0, l1_ratio=0.5)
#         model.fit(X_train, y_train)
        
#         # Predict on test data
#         y_test_pred = model.predict(X_test)
#         print(y_test_pred)
#         # 将预测结果存储在DataFrame中
#         results = pd.DataFrame({
#             'DATE': test_data['DATE'].values,
#             'permno': test_data['permno'].values,
#             'True_RET': y_test,
#             'Pred_RET': y_test_pred
#         })
        
#         # 保存到单独的CSV文件
#         results.to_csv(f'predictions_{year}.csv', index=False)
        
#         print('R^2 for {}: {}'.format(year, r2_score(results['True_RET'], results['Pred_RET']) * 100))

# with open("data/characteristics_with_ind_interaction_return.pkl", "rb") as f:
#     data = pickle.load(f)

# train_ols(data)